In [2]:
import ultralytics
import scipy
import pandas as pd
from ultralytics import YOLO
from scipy.io import loadmat
import pandas as pd
import os
import numpy as np

In [3]:
frame_train_path = "/home/dl.net/sabbasi/Documents/Face-Reid/data/archive/frame_train.mat"
ID_train_path = "/home/dl.net/sabbasi/Documents/Face-Reid/data/archive/ID_train.mat"

frame_test_path = "/home/dl.net/sabbasi/Documents/Face-Reid/data/archive/frame_test.mat"
ID_test_path = "/home/dl.net/sabbasi/Documents/Face-Reid/data/archive/ID_test.mat"

frames_dir = "/home/dl.net/sabbasi/Documents/Face-Reid/data/archive/frames/"
annotations_dir = "/home/dl.net/sabbasi/Documents/Face-Reid/data/archive/annotations/"


data_frame_train = loadmat(frame_train_path)
print(data_frame_train.keys())

data_ID_train = loadmat(ID_train_path)
print(data_ID_train.keys())

data_frame_test = loadmat(frame_test_path)
print(data_frame_test.keys())

data_ID_test = loadmat(ID_test_path)
print(data_ID_test.keys())

dict_keys(['__header__', '__version__', '__globals__', 'img_index_train'])
dict_keys(['__header__', '__version__', '__globals__', 'ID_train'])
dict_keys(['__header__', '__version__', '__globals__', 'img_index_test'])
dict_keys(['__header__', '__version__', '__globals__', 'ID_test2'])


In [4]:
df_img_index = pd.DataFrame(data_frame_train["img_index_train"].reshape(-1, 1), columns=["image_index"])
df_ID_train = pd.DataFrame(data_ID_train["ID_train"].reshape(-1, 1), columns=["ID"])
df_train = pd.concat([df_img_index, df_ID_train], axis=1)


df_img_index_test = pd.DataFrame(data_frame_test["img_index_test"].reshape(-1, 1), columns=["image_index"])
df_ID_test = pd.DataFrame(data_ID_test["ID_test2"].reshape(-1, 1), columns=["ID"])
df_test = pd.concat([df_img_index_test, df_ID_test], axis=1)


In [5]:
df_train["image_index"] = df_train["image_index"].apply(lambda x: x[0])
df_train["image_path"] = df_train["image_index"].apply(lambda x: x.replace("[", "").replace("]", "") + ".jpg").apply(lambda x : os.path.join(frames_dir, x))

def safe_load(row):
    path = os.path.join(annotations_dir, row["image_index"] + ".jpg.mat")

    if not os.path.exists(path):
        print("MISSING FILE:", path)
        return None

    data = loadmat(path)

    if "box_new" not in data:
        print("NO box_new:", path, data.keys())
        return None

    return data["box_new"]


df_train["annotations"] = df_train.apply(safe_load, axis=1)

NO box_new: /home/dl.net/sabbasi/Documents/Face-Reid/data/archive/annotations/c1s2_006041.jpg.mat dict_keys(['__header__', '__version__', '__globals__', 'anno_file'])
NO box_new: /home/dl.net/sabbasi/Documents/Face-Reid/data/archive/annotations/c2s1_049776.jpg.mat dict_keys(['__header__', '__version__', '__globals__', 'anno_file'])
NO box_new: /home/dl.net/sabbasi/Documents/Face-Reid/data/archive/annotations/c6s1_036851.jpg.mat dict_keys(['__header__', '__version__', '__globals__', 'anno_file'])


In [6]:
df_train["image_path"].iloc[0]

'/home/dl.net/sabbasi/Documents/Face-Reid/data/archive/frames/c1s1_000151.jpg'

In [7]:

# Load a model
model = YOLO("yolo26n.pt")  # load an official model

# Predict with the model
results = model(df_train["image_path"].sample(n=5).tolist())  # predict on an image

# Access the results/archive/frames/c3s1_027701'
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box


0: 384x640 5 persons, 2 bicycles, 11.1ms
1: 384x640 2 persons, 1 bicycle, 11.1ms
2: 384x640 4 persons, 1 bicycle, 11.1ms
3: 384x640 1 person, 1 bicycle, 11.1ms
4: 384x640 2 persons, 7 bicycles, 11.1ms
Speed: 2.6ms preprocess, 11.1ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)
